In [1]:
from transformers import Mistral3ForConditionalGeneration, FineGrainedFP8Config, AutoTokenizer
import torch

In [2]:
torch.cuda.is_available()

True

In [ ]:
model_id = "mistralai/Ministral-3-3B-Instruct-2512"
#tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct", cache_dir="/hpcstor6/scratch01/h/huuthanhvy.nguyen001")

In [7]:
model = Mistral3ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="cuda:0",
    quantization_config=FineGrainedFP8Config(dequantize=True),
    cache_dir="/hpcstor6/scratch01/h/huuthanhvy.nguyen001"

)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.67G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

In [8]:
model

Mistral3ForConditionalGeneration(
  (model): Mistral3Model(
    (vision_tower): PixtralVisionModel(
      (patch_conv): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
      (ln_pre): PixtralRMSNorm((1024,), eps=1e-05)
      (transformer): PixtralTransformer(
        (layers): ModuleList(
          (0-23): 24 x PixtralAttentionLayer(
            (attention_norm): PixtralRMSNorm((1024,), eps=1e-05)
            (feed_forward): PixtralMLP(
              (gate_proj): Linear(in_features=1024, out_features=4096, bias=False)
              (up_proj): Linear(in_features=1024, out_features=4096, bias=False)
              (down_proj): Linear(in_features=4096, out_features=1024, bias=False)
              (act_fn): SiLUActivation()
            )
            (attention): PixtralAttention(
              (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
              (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
              (q_proj): Linear(in_f

In [10]:
import json

with open ("/home/huuthanhvy.nguyen001/newmodel/a1_gen_chatgpt_p0.json", "r") as file:
    data1 = json.load(file)

with open ("/home/huuthanhvy.nguyen001/newmodel/a1_tune_chatgpt_p0.json", "r") as file:
    data2 = json.load(file)

In [11]:
generate_essay = data1["result"]
tune_essay = data2["result"]

In [12]:
rubric = {
    "issues": {
        "4": "Issue is stated clearly and described comprehensively, delivering all relevant information necessary for full understanding.",
        "3": "Issue is stated, described, and clarified so that understanding is not seriously impeded by omissions.",
        "2": "Issue is stated but description leaves some terms undefined, ambiguities unexplored, boundaries undetermined, and/or backgrounds unknown.",
        "1": "Issue is stated without clarification or description."
    },
    "evidence": {
        "4": "Information is taken from source(s) with enough interpretation/evaluation to develop a comprehensive analysis or synthesis. Viewpoints of experts are questioned thoroughly.",
        "3": "Information is taken from source(s) with enough interpretation/evaluation to develop a coherent analysis or synthesis. Viewpoints of experts are subject to questioning.",
        "2": "Information is taken from source(s) with some interpretation/evaluation, but not enough to develop a coherent analysis or synthesis. Viewpoints of experts are taken as mostly fact, with little questioning.",
        "1": "Information is taken from source(s) without any interpretation/evaluation. Viewpoints of experts are taken as fact, without question."
    },
    "assumptions": {
        "4": "Thoroughly (systematically and methodically) analyzes own and others' assumptions and carefully evaluates the relevance of contexts when presenting a position.",
        "3": "Identifies own and others' assumptions and several relevant contexts when presenting a position. Questions some assumptions.",
        "2": "Identifies several relevant contexts when presenting a position. May be more aware of others' assumptions than one's own. Shows emerging awareness of present assumptions.",
        "1": "Shows an emerging awareness of present assumptions (sometimes labels assertions as assumptions). Begins to identify some contexts when presenting a position."
    },
    "position": {
        "4": "Specific position is imaginative, taking into account the complexities of an issue. Limits of position are acknowledged. Others' points of view are synthesized within position.",
        "3": "Specific position takes into account the complexities of an issue. Others' points of view are acknowledged within position.",
        "2": "Specific position acknowledges different sides of an issue.",
        "1": "Specific position is stated, but is simplistic and obvious."
    },
    "conclusions": {
        "4": "Conclusions and related outcomes are logical and reflect student's informed evaluation and ability to place evidence and perspectives discussed in priority order.",
        "3": "Conclusion is logically tied to a range of information, including opposing viewpoints; related outcomes are identified clearly.",
        "2": "Conclusion is logically tied to information (chosen to fit the desired conclusion); some related outcomes are identified clearly.",
        "1": "Conclusion is inconsistently tied to some of the information discussed; related outcomes are oversimplified."
    }
}


In [13]:
messages1 = [
    {
        "role": "system",
        "content": f""" Score the essay on these 5 dimensions, each from 1 to 4:

Rubric:
{json.dumps(rubric, indent=2)}

Scoring scale:
1 = Benchmark
2 = Milestone (lower)
3 = Milestone (upper)
4 = Capstone

Return ONLY a JSON object, no explanation:
{{"issues": <1-4>, "evidence": <1-4>, "assumptions": <1-4>, "position": <1-4>, "conclusions": <1-4>}}"""
    },
    {
        "role": "user",
        "content": f"""Score this essay:\n\n{generate_essay}"""
    },
]

In [18]:
messages2 = [
    {
        "role": "system",
        "content": f""" Score the essay on these 5 dimensions, each from 1 to 4:

Rubric:
{json.dumps(rubric, indent=2)}

Scoring scale:
1 = Benchmark
2 = Milestone (lower)
3 = Milestone (upper)
4 = Capstone

Return ONLY a JSON object, no explanation:
{{"issues": <1-4>, "evidence": <1-4>, "assumptions": <1-4>, "position": <1-4>, "conclusions": <1-4>}}"""
    },
    {
        "role": "user",
        "content": f"""Score this essay:\n\n{tune_essay}"""
    },
]

In [15]:
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    cache_dir="/hpcstor6/scratch01/h/huuthanhvy.nguyen001"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

In [16]:
inputs1 = tokenizer.apply_chat_template(
	messages1,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

In [19]:
inputs2 = tokenizer.apply_chat_template(
	messages2,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

In [23]:
outputs1 = model.generate(**inputs1, max_new_tokens=100)
outputs2 = model.generate(**inputs2, max_new_tokens=100)

Both `max_new_tokens` (=100) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [24]:
print(tokenizer.decode(outputs1[0][inputs1["input_ids"].shape[-1]:]))  

```json
{
  "issues": 4,
  "evidence": 4,
  "assumptions": 3,
  "position": 4,
  "conclusions": 4
}
```</s>


In [25]:
print(tokenizer.decode(outputs2[0][inputs2["input_ids"].shape[-1]:]))  

```json
{
  "issues": 4,
  "evidence": 4,
  "assumptions": 4,
  "position": 4,
  "conclusions": 4
}
```</s>
